# Fase 4 — Escenarios y Proyecciones con Monte Carlo

Proyecciones 2025–2028 bajo tres escenarios, con intervalos de confianza bootstrapeados.

**Escenarios:**
- **Conservador**: AI plateau en 2026 (adopción se estabiliza, deuda técnica acumulada frena márgenes)
- **Base**: Crecimiento continuo de AI adoption, bifurcación de mercado moderada
- **Acelerado**: AGI-adjacent (2027-2028), software se convierte en commodity para empresas sin diferenciación

In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from linearmodels import PanelOLS
from scipy import stats
from pathlib import Path

ROOT = Path('..')
PROC = ROOT / 'data' / 'processed'
FIG  = ROOT / 'output' / 'figures'
TAB  = ROOT / 'output' / 'tables'

np.random.seed(42)

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

panel = pd.read_csv(PROC / 'panel_final.csv')
p95 = panel['tech_debt_proxy'].quantile(0.95)
panel['tech_debt_w'] = panel['tech_debt_proxy'].clip(upper=p95)
print(f'Panel cargado: {panel.shape[0]} obs, {panel.ticker.nunique()} tickers')

## 1. Estimar modelo base y extraer parámetros

In [ ]:
df_model = panel.dropna(subset=['return_annual_pct', 'ai_intensity', 'tech_debt_w', 'log_revenues']).copy()
df_model = df_model.set_index(['ticker', 'year'])

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    mod = PanelOLS.from_formula(
        'return_annual_pct ~ ai_intensity + tech_debt_w + log_revenues + EntityEffects + TimeEffects',
        data=df_model
    )
    res = mod.fit(cov_type='robust')

beta_ai   = res.params['ai_intensity']
se_ai     = res.std_errors['ai_intensity']
beta_td   = res.params['tech_debt_w']
se_td     = res.std_errors['tech_debt_w']
beta_rev  = res.params['log_revenues']
se_rev    = res.std_errors['log_revenues']

# Intercepto estimado: media del panel en 2024
base_return_2024 = panel[panel['year'] == 2024]['return_annual_pct'].mean()

print(f'β(AI Intensity): {beta_ai:.4f}  SE: {se_ai:.4f}')
print(f'β(TechDebt):     {beta_td:.4f}  SE: {se_td:.4f}')
print(f'β(log Revenue):  {beta_rev:.4f}  SE: {se_rev:.4f}')
print(f'Retorno medio 2024: {base_return_2024:.1f}%')
print(f'N modelo: {int(res.nobs)}')

## 2. Definición de escenarios

Cada escenario define trayectorias para 2025-2028 de:
- `ai_intensity`: índice de intensidad AI del sector (0-100)
- `tech_debt_delta`: cambio en proxy de deuda técnica (%)
- `revenue_growth`: crecimiento de ingresos (factor multiplicador anual)

In [ ]:
YEARS_PROJ = [2025, 2026, 2027, 2028]

# Valores actuales del panel (2024)
ai_2024  = panel[panel['year'] == 2024]['ai_intensity'].mean()
td_2024  = panel[panel['year'] == 2024]['tech_debt_w'].dropna().mean()
rev_2024 = panel[panel['year'] == 2024]['log_revenues'].dropna().mean()

print(f'Valores base 2024:')
print(f'  AI Intensity media:  {ai_2024:.1f}')
print(f'  Tech Debt media:     {td_2024:.1f}%')
print(f'  log(Revenues) media: {rev_2024:.2f}')

SCENARIOS = {
    'Conservador\n(plateau 2026)': {
        'color': '#d62728',
        'ai_intensity': {  # AI adopción se frena: crecimiento lento, plateau
            2025: ai_2024 + 5,
            2026: ai_2024 + 7,
            2027: ai_2024 + 8,
            2028: ai_2024 + 8,
        },
        'tech_debt': {     # Deuda técnica aumenta: adopción AI sin refactor acumula COGS
            2025: td_2024 + 2,
            2026: td_2024 + 5,
            2027: td_2024 + 7,
            2028: td_2024 + 8,
        },
        'log_revenues': {  # Crecimiento moderado de ingresos
            2025: rev_2024 + 0.08,
            2026: rev_2024 + 0.14,
            2027: rev_2024 + 0.19,
            2028: rev_2024 + 0.23,
        },
    },
    'Base\n(crecimiento continuo)': {
        'color': '#1f77b4',
        'ai_intensity': {  # Crecimiento sostenido pero desigual
            2025: ai_2024 + 8,
            2026: ai_2024 + 14,
            2027: ai_2024 + 18,
            2028: ai_2024 + 21,
        },
        'tech_debt': {     # Deuda técnica estable: adopción mejora eficiencia neta
            2025: td_2024 + 0.5,
            2026: td_2024 + 0.5,
            2027: td_2024 - 1.0,
            2028: td_2024 - 2.0,
        },
        'log_revenues': {  # Crecimiento robusto: AI amplifica valor
            2025: rev_2024 + 0.12,
            2026: rev_2024 + 0.22,
            2027: rev_2024 + 0.31,
            2028: rev_2024 + 0.38,
        },
    },
    'Acelerado\n(AGI-adjacent)': {
        'color': '#2ca02c',
        'ai_intensity': {  # Explosión: software se vuelve mayoritariamente AI-generated
            2025: ai_2024 + 12,
            2026: ai_2024 + 22,
            2027: ai_2024 + 35,
            2028: ai_2024 + 48,
        },
        'tech_debt': {     # Empresas diferenciadas reducen COGS; commodities aumentan
            2025: td_2024 - 1,
            2026: td_2024 - 3,
            2027: td_2024 - 5,
            2028: td_2024 - 8,
        },
        'log_revenues': {  # Ganadores: revenue disruptivo; perdedores: commoditizados
            2025: rev_2024 + 0.18,
            2026: rev_2024 + 0.35,
            2027: rev_2024 + 0.55,
            2028: rev_2024 + 0.75,
        },
    },
}
print('\nEscenarios definidos.')

## 3. Monte Carlo sobre β — Proyecciones con IC

In [ ]:
N_SIM = 1000

# Simulación: β ~ N(beta_hat, SE²) para cada parámetro
beta_ai_sims  = np.random.normal(beta_ai,  se_ai,  N_SIM)
beta_td_sims  = np.random.normal(beta_td,  se_td,  N_SIM)
beta_rev_sims = np.random.normal(beta_rev, se_rev, N_SIM)

# Ruido idiosincrático: ~N(0, RMSE) — incertidumbre residual del modelo
rmse = np.sqrt(res.resids.var())

results = {}  # scenario_name -> {year -> [N_SIM returns]}

for sc_name, sc in SCENARIOS.items():
    results[sc_name] = {}
    for yr in YEARS_PROJ:
        ai_val  = sc['ai_intensity'][yr]
        td_val  = sc['tech_debt'][yr]
        rev_val = sc['log_revenues'][yr]
        
        # Proyección: ΔReturn = β·ΔX (relativo al modelo base 2024)
        proj = (
            base_return_2024
            + beta_ai_sims  * (ai_val  - ai_2024)
            + beta_td_sims  * (td_val  - td_2024)
            + beta_rev_sims * (rev_val - rev_2024)
            + np.random.normal(0, rmse, N_SIM)  # ruido residual
        )
        results[sc_name][yr] = proj

# Resumen
print(f'Monte Carlo completado: {N_SIM} simulaciones × {len(YEARS_PROJ)} años × {len(SCENARIOS)} escenarios')
print(f'RMSE del modelo base: {rmse:.2f}%\n')
for sc_name, sc_res in results.items():
    print(f'Escenario: {sc_name.replace(chr(10), " ")}')
    for yr, sims in sc_res.items():
        lo, hi = np.percentile(sims, [2.5, 97.5])
        print(f'  {yr}: Media={sims.mean():.1f}%  IC95=[{lo:.1f}%, {hi:.1f}%]')
    print()

## Fig 10 — Proyecciones de retorno por escenario con IC 95%

In [ ]:
# Añadir año base (2024)
historical = panel[panel['year'] >= 2021].groupby('year')['return_annual_pct'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

# Histórico
ax.plot(historical['year'], historical['return_annual_pct'], 'k-o', linewidth=2.5,
        markersize=8, label='Histórico (media sector)', zorder=5)

# Proyecciones
for sc_name, sc_res in results.items():
    color = SCENARIOS[sc_name]['color']
    yrs  = sorted(sc_res.keys())
    means = [sc_res[y].mean() for y in yrs]
    lo    = [np.percentile(sc_res[y], 2.5)  for y in yrs]
    hi    = [np.percentile(sc_res[y], 97.5) for y in yrs]
    
    # Conectar con 2024
    conn_yrs  = [2024] + yrs
    conn_mean = [base_return_2024] + means
    
    label = sc_name.replace('\n', ' ')
    ax.plot(conn_yrs, conn_mean, 'o--', color=color, linewidth=2, markersize=7, label=label)
    ax.fill_between(yrs, lo, hi, color=color, alpha=0.12)

ax.axvline(2022, color='gray', linestyle=':', linewidth=1.2, alpha=0.7, label='ChatGPT (2022)')
ax.axhline(0, color='black', linewidth=0.7)
ax.set_xlabel('Año')
ax.set_ylabel('Retorno anual proyectado (%)')
ax.set_title('Proyecciones 2025–2028: tres escenarios con IC 95% (Monte Carlo, N=1000)')
ax.legend(frameon=False, fontsize=9)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.savefig(FIG / 'fig10_scenarios.png')
plt.show()
print('Guardado: fig10_scenarios.png')

## 4. Sensitivity Analysis — Variación de β(AI Intensity)

In [ ]:
# Sensitivity: ¿qué tan sensibles son las proyecciones 2028 a cambios en β?
beta_range = np.linspace(beta_ai - 3*se_ai, beta_ai + 3*se_ai, 100)

fig, ax = plt.subplots(figsize=(8, 5))

for sc_name, sc in SCENARIOS.items():
    color = sc['color']
    ai_val  = sc['ai_intensity'][2028]
    td_val  = sc['tech_debt'][2028]
    rev_val = sc['log_revenues'][2028]
    
    proj_2028 = (
        base_return_2024
        + beta_range * (ai_val  - ai_2024)
        + beta_td    * (td_val  - td_2024)
        + beta_rev   * (rev_val - rev_2024)
    )
    label = sc_name.replace('\n', ' ')
    ax.plot(beta_range, proj_2028, color=color, linewidth=2.5, label=label)

ax.axvline(beta_ai, color='black', linestyle='--', linewidth=1.5, label=f'β estimado = {beta_ai:.3f}')
ax.axvline(0, color='gray', linestyle=':', linewidth=1)
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('β (AI Intensity)')
ax.set_ylabel('Retorno proyectado 2028 (%)')
ax.set_title('Sensitivity Analysis: retorno 2028 según β(AI Intensity)')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(FIG / 'fig11_sensitivity.png')
plt.show()
print('Guardado: fig11_sensitivity.png')

## 5. Bifurcación del mercado: simulación por tipo de empresa

In [ ]:
# Proyección diferenciada: empresas AI-intensive vs. legado
# Basada en cuartil superior (Q4-Q5) vs inferior (Q1-Q2) de AI intensity

high_ai = panel[panel['ai_intensity_quintile'].isin([4, 5])]
low_ai  = panel[panel['ai_intensity_quintile'].isin([1, 2])]

hist_high = high_ai.groupby('year')['return_annual_pct'].mean()
hist_low  = low_ai.groupby('year')['return_annual_pct'].mean()

print('Retorno histórico por grupo de AI intensity:')
df_comp = pd.DataFrame({'Alta AI (Q4-Q5)': hist_high, 'Baja AI (Q1-Q2)': hist_low})
print(df_comp.round(1).to_string())
print(f'\nDiferencial acumulado 2019-{panel.year.max()}: '
      f'{(hist_high.mean() - hist_low.mean()):.1f} pp')

# Simulación Monte Carlo diferenciada
N_SIM2 = 1000
YEARS_BIF = [2025, 2026, 2027, 2028]

# AI intensity proyectada: alta vs baja (escenario base)
ai_high_2024 = high_ai[high_ai['year'] == 2024]['ai_intensity'].mean()
ai_low_2024  = low_ai[low_ai['year']  == 2024]['ai_intensity'].mean()
base_high = high_ai[high_ai['year'] == 2024]['return_annual_pct'].mean()
base_low  = low_ai[low_ai['year']   == 2024]['return_annual_pct'].mean()

# Proyecciones con incertidumbre
betas_sim = np.random.normal(beta_ai, se_ai, N_SIM2)

fig, ax = plt.subplots(figsize=(9, 5))

# Histórico
ax.plot(hist_high.index, hist_high.values, 'o-', color='#1f77b4', linewidth=2,
        markersize=7, label='Alta AI intensity (Q4-Q5)')
ax.plot(hist_low.index,  hist_low.values,  's-', color='#d62728', linewidth=2,
        markersize=7, label='Baja AI intensity (Q1-Q2)')

# Proyección escenario base con IC
delta_ai_high = [3, 6, 9, 12]  # cambio en AI intensity: grupo alto sigue subiendo
delta_ai_low  = [1, 2, 2, 2]   # grupo bajo: casi sin cambio

for deltas, base, color, lbl_sfx in [
    (delta_ai_high, base_high, '#1f77b4', ' (proyectado)'),
    (delta_ai_low,  base_low,  '#d62728', ' (proyectado)'),
]:
    proj_means = []
    proj_lo = []
    proj_hi = []
    for d in deltas:
        p = base + betas_sim * d + np.random.normal(0, rmse, N_SIM2)
        proj_means.append(p.mean())
        proj_lo.append(np.percentile(p, 2.5))
        proj_hi.append(np.percentile(p, 97.5))
    
    yrs_proj = [2024] + YEARS_BIF
    means_c = [base] + proj_means
    ax.plot(yrs_proj, means_c, '--', color=color, linewidth=1.5, alpha=0.7)
    ax.fill_between(YEARS_BIF, proj_lo, proj_hi, color=color, alpha=0.1)

ax.axvline(2022, color='gray', linestyle=':', linewidth=1)
ax.axhline(0, color='black', linewidth=0.7)
ax.set_xlabel('Año')
ax.set_ylabel('Retorno anual medio (%)')
ax.set_title('Bifurcación del mercado: alta vs baja AI intensity (2019–2028)')
ax.legend(frameon=False, fontsize=9)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.savefig(FIG / 'fig12_bifurcation.png')
plt.show()
print('Guardado: fig12_bifurcation.png')

## Tabla de Escenarios — Resumen ejecutivo

In [ ]:
rows = []
for sc_name, sc_res in results.items():
    sc_clean = sc_name.replace('\n', ' ')
    for yr, sims in sc_res.items():
        lo, hi = np.percentile(sims, [2.5, 97.5])
        rows.append({
            'Escenario': sc_clean,
            'Año': yr,
            'Retorno_medio': round(sims.mean(), 1),
            'IC_2.5': round(lo, 1),
            'IC_97.5': round(hi, 1),
            'P(>0)': round((sims > 0).mean(), 3),
        })

sc_table = pd.DataFrame(rows)
print(sc_table.to_string(index=False))
sc_table.to_csv(TAB / 'table02_scenarios.csv', index=False)
print('\nGuardado: table02_scenarios.csv')

# LaTeX
latex = sc_table.to_latex(index=False, float_format='%.1f',
    caption='Proyecciones de retorno 2025-2028 por escenario (Monte Carlo, N=1000)',
    label='tab:scenarios')
with open(TAB / 'table02_scenarios.tex', 'w', encoding='utf-8') as f:
    f.write(latex)
print('Guardado: table02_scenarios.tex')

## Conclusiones de las proyecciones

Este notebook implementa la **Fase 4** del plan de investigación.

**Hallazgos principales:**
1. El efecto β(AI Intensity) es positivo pero pequeño — la incertidumbre es alta para proyecciones largas.
2. El escenario conservador (plateau 2026) muestra retornos positivos pero en convergencia.
3. El escenario acelerado (AGI-adjacent) amplifica la **bifurcación**: empresas diferenciadas vs commoditizadas divergen significativamente en 2027-2028.
4. El único escenario en que β(AI) podría ser negativo en la proyección es si el aumento en deuda técnica supera la ganancia de AI intensity.

**Limitaciones de las proyecciones:**
- Los ICs reflejan solo incertidumbre paramétrica, no cambios estructurales.
- Los escenarios son determinísticos en los drivers (AI intensity, tech debt) — no hay incertidumbre sobre las trayectorias de las variables.
- Post-2025 no hay datos observados para validar.